# ControlNet Training — Dong Ho (theo `docs/train.md` của tác giả)

Notebook bám sát hướng dẫn gốc tại https://github.com/lllyasviel/ControlNet/blob/main/docs/train.md.

Pipeline 4 bước (như docs):
1. Get a dataset — đã có ở `training/dongho/{source,target}` + `prompt.json`.
2. Load the dataset — `tutorial_dataset.py` (cell `## 6`).
3. What SD model do you want to control? → SD 1.5, chạy `tool_add_control.py` (cell `## 5`).
4. Train! — `tutorial_train.py` (cell `## 7`).

Hyperparameter mặc định lấy từ docs: `batch_size=4, sd_locked=True, only_mid_control=False, learning_rate=1e-5, precision=32, logger_freq=300`. Đã giảm `batch_size=1 + accumulate_grad_batches=4` để vừa T4 16 GB như docs gợi ý ở phần OOM.

## Prerequisites (làm local TRƯỚC khi chạy)

1. Zip dataset đã augment:
   ```bash
   cd ~/Desktop/ControlNet/ControlNet/training && zip -r dongho.zip dongho/
   ```
2. Upload `dongho.zip` lên Google Drive root (`MyDrive/dongho.zip`).
3. Trong Colab: **Runtime → Change runtime type → GPU (T4)**.

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo + cài deps + patch tương thích PL 2.x / torch 2.6+

In [ ]:
%cd /content
!rm -rf ControlNet
!git clone https://github.com/lllyasviel/ControlNet.git
%cd /content/ControlNet

In [ ]:
!pip install -q "pytorch-lightning>=2.0,<3" omegaconf einops open_clip_torch \
  transformers timm safetensors ftfy opencv-python

In [ ]:
import pathlib

# 1) PL 2.x: rank_zero_only đổi đường dẫn import
PATCH_OLD = 'from pytorch_lightning.utilities.distributed import rank_zero_only'
PATCH_NEW = ('try:\n'
             '    from pytorch_lightning.utilities.distributed import rank_zero_only\n'
             'except ImportError:\n'
             '    from pytorch_lightning.utilities.rank_zero import rank_zero_only')
for path in ['/content/ControlNet/cldm/logger.py',
             '/content/ControlNet/ldm/models/diffusion/ddpm.py']:
    p = pathlib.Path(path); s = p.read_text()
    if PATCH_OLD in s and PATCH_NEW not in s:
        p.write_text(s.replace(PATCH_OLD, PATCH_NEW))
        print('patched PL import:', path)

# 2) torch 2.6+: weights_only=True default vỡ ckpt Lightning
p = pathlib.Path('/content/ControlNet/cldm/model.py'); s = p.read_text()
old = 'state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location)))'
new = 'state_dict = get_state_dict(torch.load(ckpt_path, map_location=torch.device(location), weights_only=False))'
if old in s and new not in s:
    p.write_text(s.replace(old, new))
    print('patched cldm/model.py')

# 3) tool_add_control.py: thêm mmap=True + weights_only=False để giảm peak RAM Colab free
p = pathlib.Path('/content/ControlNet/tool_add_control.py'); s = p.read_text()
old = 'pretrained_weights = torch.load(input_path)'
new = 'pretrained_weights = torch.load(input_path, weights_only=False, mmap=True)'
if old in s and new not in s:
    p.write_text(s.replace(old, new))
    print('patched tool_add_control.py (mmap=True)')

## 3. Mount Drive + giải nén dataset (Step 1 — Get a dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/ControlNet/training
!unzip -q /content/drive/MyDrive/dongho.zip -d /content/ControlNet/training/
!echo 'target:' && ls /content/ControlNet/training/dongho/target/ | wc -l
!echo 'source:' && ls /content/ControlNet/training/dongho/source/ | wc -l
!echo 'prompt.json lines:' && wc -l /content/ControlNet/training/dongho/prompt.json

## 4. Patch dataset loader (Step 2 — Load the dataset)

Ghi đè `tutorial_dataset.py` đúng theo template trong docs (chỉ đổi đường dẫn `fill50k` → `dongho`).

In [ ]:
from pathlib import Path
Path('/content/ControlNet/tutorial_dataset.py').write_text('''import json
import cv2
import numpy as np

from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self):
        self.data = []
        with open('./training/dongho/prompt.json', 'rt') as f:
            for line in f:
                self.data.append(json.loads(line))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        source_filename = item['source']
        target_filename = item['target']
        prompt = item['prompt']

        source = cv2.imread('./training/dongho/' + source_filename)
        target = cv2.imread('./training/dongho/' + target_filename)

        # Do not forget that OpenCV read images in BGR order.
        source = cv2.cvtColor(source, cv2.COLOR_BGR2RGB)
        target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

        # Normalize source images to [0, 1].
        source = source.astype(np.float32) / 255.0

        # Normalize target images to [-1, 1].
        target = (target.astype(np.float32) / 127.5) - 1.0

        return dict(jpg=target, txt=prompt, hint=source)
''')
print('Đã ghi tutorial_dataset.py')

In [ ]:
# tutorial_dataset_test.py (theo docs)
%cd /content/ControlNet
from tutorial_dataset import MyDataset

dataset = MyDataset()
print(len(dataset))

item = dataset[0]
print('prompt:', item['txt'])
print('jpg shape:', item['jpg'].shape)
print('hint shape:', item['hint'].shape)

## 5. Tải SD 1.5 + chạy `tool_add_control.py` (Step 3 — What SD model do you want to control)

Giống docs: `python tool_add_control.py ./models/v1-5-pruned.ckpt ./models/control_sd15_ini.ckpt`.

In [ ]:
!mkdir -p /content/ControlNet/models
!wget -c https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt \
  -O /content/ControlNet/models/v1-5-pruned.ckpt

In [ ]:
%cd /content/ControlNet
import os
OUTPUT = '/content/ControlNet/models/control_sd15_ini.ckpt'
if os.path.exists(OUTPUT) and os.path.getsize(OUTPUT) > 5e9:
    print('Đã có', OUTPUT, '— bỏ qua')
else:
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)  # tool_add_control.py assert output not exist
    !python tool_add_control.py ./models/v1-5-pruned.ckpt ./models/control_sd15_ini.ckpt
    assert os.path.exists(OUTPUT) and os.path.getsize(OUTPUT) > 5e9, 'Tạo init ckpt thất bại'

## 6. Training script (Step 4 — Train!)

Theo template `tutorial_train.py` của docs, chỉ điều chỉnh:
- `gpus=1` (PL 1.x) → `accelerator='gpu', devices=1` (PL 2.x)
- thêm `accumulate_grad_batches=4 + batch_size=1 + precision=16` để vừa T4 (theo gợi ý OOM ở docs)
- thêm `ModelCheckpoint` callback lưu định kỳ

In [ ]:
from pathlib import Path
Path('/content/ControlNet/tutorial_train.py').write_text('''from share import *

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
from tutorial_dataset import MyDataset
from cldm.logger import ImageLogger
from cldm.model import create_model, load_state_dict


# Configs (giong docs/train.md, dieu chinh cho T4)
resume_path = './models/control_sd15_ini.ckpt'
batch_size = 1
logger_freq = 300
learning_rate = 1e-5
sd_locked = True
only_mid_control = False
max_steps = 5000


# First use cpu to load models. Pytorch Lightning will automatically move it to GPUs.
model = create_model('./models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(resume_path, location='cpu'))
model.learning_rate = learning_rate
model.sd_locked = sd_locked
model.only_mid_control = only_mid_control


# Misc
dataset = MyDataset()
dataloader = DataLoader(dataset, num_workers=0, batch_size=batch_size, shuffle=True)
logger = ImageLogger(batch_frequency=logger_freq)
ckpt_cb = ModelCheckpoint(
    dirpath='./training/dongho_ckpts',
    filename='dongho-{step:06d}',
    every_n_train_steps=1000,
    save_top_k=-1,
    save_last=True,
)
trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision=16,
    accumulate_grad_batches=4,
    max_steps=max_steps,
    callbacks=[logger, ckpt_cb],
    default_root_dir='./training/dongho_ckpts',
)


# Train!
trainer.fit(model, dataloader)
''')
print('Đã ghi tutorial_train.py')

## 7. Train

Giống docs: `!python tutorial_train.py`. Trên T4 free: ~2–3 s/step → 5000 step ≈ 3–4 h.
Có thể `Interrupt` bất cứ lúc nào — checkpoint định kỳ đã lưu trong `training/dongho_ckpts/`.

In [ ]:
%cd /content/ControlNet
!python tutorial_train.py

## 8. Backup checkpoints về Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/dongho_ckpts
!cp -n /content/ControlNet/training/dongho_ckpts/*.ckpt /content/drive/MyDrive/dongho_ckpts/ || true
!ls -lh /content/drive/MyDrive/dongho_ckpts/

## 9. (Tuỳ chọn) Inference test

Sau khi train xong, test với một sketch (nét đen trên nền trắng) đặt tại `/content/my_sketch.png`.

In [ ]:
import os, cv2, numpy as np, torch, einops
from pytorch_lightning import seed_everything
from cldm.model import create_model, load_state_dict
from cldm.ddim_hacked import DDIMSampler

CKPT = '/content/ControlNet/training/dongho_ckpts/last.ckpt'
SKETCH_PATH = '/content/my_sketch.png'
USER_INPUT_IS_BLACK_ON_WHITE = True
RES = 512
PROMPT   = 'Vietnamese Dong Ho folk woodblock painting, warrior riding a beast'
A_PROMPT = 'best quality, extremely detailed, traditional woodblock print'
N_PROMPT = 'blurry, low quality, deformed'

assert os.path.exists(CKPT), f'Không thấy {CKPT}. Chạy cell ## 7 (Train) trước.'
assert os.path.exists(SKETCH_PATH), f'Không thấy sketch {SKETCH_PATH}.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = create_model('/content/ControlNet/models/cldm_v15.yaml').cpu()
model.load_state_dict(load_state_dict(CKPT, location=device))
model = model.to(device)
sampler = DDIMSampler(model)

sketch = cv2.cvtColor(cv2.imread(SKETCH_PATH), cv2.COLOR_BGR2RGB)
sketch = cv2.resize(sketch, (RES, RES))
gray = cv2.cvtColor(sketch, cv2.COLOR_RGB2GRAY)
_, mask = cv2.threshold(gray, 127, 255,
                        cv2.THRESH_BINARY_INV if USER_INPUT_IS_BLACK_ON_WHITE else cv2.THRESH_BINARY)
scribble = cv2.cvtColor(mask, cv2.COLOR_GRAY2RGB)

with torch.no_grad():
    control = torch.from_numpy(scribble.copy()).float().to(device) / 255.0
    control = einops.rearrange(control, 'h w c -> 1 c h w')
    seed_everything(42)
    cond    = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([PROMPT + ', ' + A_PROMPT])]}
    un_cond = {'c_concat': [control], 'c_crossattn': [model.get_learned_conditioning([N_PROMPT])]}
    shape = (4, RES // 8, RES // 8)
    model.control_scales = [1.0] * 13
    samples, _ = sampler.sample(20, 1, shape, cond, verbose=False, eta=0.0,
                                unconditional_guidance_scale=9.0,
                                unconditional_conditioning=un_cond)
    x = model.decode_first_stage(samples)
    x = (einops.rearrange(x, 'b c h w -> b h w c') * 127.5 + 127.5).cpu().numpy().clip(0, 255).astype(np.uint8)

from PIL import Image
Image.fromarray(x[0]).save('/content/test_output.png')
print('Saved to /content/test_output.png')
from IPython.display import Image as IPImg
IPImg('/content/test_output.png')